# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List the available record set @id values and summarize included fields/columns
record_sets = dataset.metadata.record_sets
print(f"Record sets found: {[rs.id for rs in record_sets]}")

for record_set in record_sets:
    print(f"\nRecord set: {record_set.id}")
    print(f"Name: {getattr(record_set, 'name', None)}")
    print("Fields and columns (with @id):")
    for field in getattr(record_set, 'fields', []):
        print(f"  Field: {field.id} | Name: {getattr(field, 'name', None)} | Data type: {getattr(field, 'data_type', None)}")
    for column in getattr(record_set, 'columns', []):
        print(f"  Column: {column.id} | Name: {getattr(column, 'name', None)} | Data type: {getattr(column, 'data_type', None)}")

## 3. Data Extraction
Load data from all record sets as DataFrames for analysis. Use the record set and field/column `@id`s from the overview.

In [ ]:
# Extract records for each record set
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set {rs_id} with shape {df.shape}")

# Display columns of the first record set as an example
if len(record_set_ids) > 0:
    example_rs = record_set_ids[0]
    print(f"\nColumns for {example_rs}: {dataframes[example_rs].columns.tolist()}")
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps such as filtering, normalization, outlier removal, and grouping. Ensure all entities are referenced by `@id`.

In [ ]:
# Select a record set with numeric columns for analysis
# You may need to adjust the IDs based on the printout in section 2.

# Example selection:
record_set_id = record_set_ids[0]  # Use the first record set as example
df = dataframes[record_set_id]

print(f"Examining DataFrame for record set: {record_set_id}")
print(df.dtypes)  # Show types to select a numeric field

numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields: {numeric_fields}")

# Pick the first available numeric field
if len(numeric_fields) == 0:
    print("No numeric fields found for processing.")
else:
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].mean()  # Use mean as simple threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field if available
    non_numeric_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
    group_field = non_numeric_fields[0] if non_numeric_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field and/or relationships between fields using matplotlib.

In [ ]:
import matplotlib.pyplot as plt

if len(record_set_ids) > 0 and len(numeric_fields) > 0:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field} in {record_set_id}')
    plt.show()

    # If both normalized field and a group field exist, plot grouped means
    if 'filtered_df' in locals() and group_field is not None:
        grouped_df.plot(x=group_field, y=numeric_field, kind='bar', figsize=(10,4))
        plt.ylabel(numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.show()

## 6. Conclusion
In this notebook, you loaded, explored, and performed basic data processing and visualization for the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the Croissant schema and the `mlcroissant` library.

You can further extend this analysis by selecting other record sets and fields by their `@id`, or by building custom analysis pipelines on top of the loaded DataFrames.